# **Filling Missing County GDP per Capita Values**

This notebook fills missing **county-level GDP per capita** values in the original `GDP_percapita.csv` used for the solar propensity score pipeline.

### Data used
- `../Solar-NIMBY/data/GDP_percapita.csv` — original GDP-per-capita file
- `missing_gdp_per_capita_counties.csv` — counties missing GDP-per-capita values
- `data/CAGDP2/` — BEA 2021 county GDP files
- `data/co-est2021-alldata.csv` — Census 2021 county population estimates

### Method

For each missing county, the notebook:
1. patches or recovers the correct **GEOID**
2. matches **2021 GDP** from BEA CAGDP2
3. matches **2021 population** from Census
4. computes:

$$
\text{GDP per capita} = \frac{\text{2021 GDP} \times 1000}{\text{2021 population}}
$$

BEA GDP is reported in **thousands of dollars**, so it is multiplied by 1000 before dividing by population.

### Special cases
- **Virginia:** some counties/cities are reported by BEA as combined areas, so a manual crosswalk is used
- **Maui:** filled using the combined **Maui + Kalawao** BEA area
- **Puerto Rico:** unresolved municipio rows are left identifiable in the audit output unless a separate source or proxy is chosen

### Outputs
- `data/processed/GDP_percapita_merged.csv` — final merged GDP-per-capita file
- `data/processed/GDP_percapita_fill_audit_2021.csv` — audit file showing fills and unresolved rows

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

scores_root = Path(".")  # solar-county-propensity-scores repo root

nimby_candidates = [
    Path("../Solar-NIMBY/data"),
    Path("../solar-NIMBY/data"),
    Path("../../Solar-NIMBY/data"),
    Path("../../solar-NIMBY/data"),
]
nimby_data = next((p for p in nimby_candidates if p.exists()), None)
if nimby_data is None:
    raise FileNotFoundError("Could not find sibling Solar-NIMBY/data folder.")

orig_gdp_path = nimby_data / "GDP_percapita.csv"
missing_path = scores_root / "missing_gdp_per_capita_counties.csv"
bea_dir = scores_root / "data" / "CAGDP2"

co_est_path = scores_root / "data" / "co-est2021-alldata.csv"
acs_path = scores_root / "data" / "ACSDT1Y2021.B01003_2026-03-12T195123" / "ACSDT1Y2021.B01003-Data.csv"

out_path = scores_root / "data" / "processed" / "GDP_percapita_merged.csv"
audit_path = scores_root / "data" / "processed" / "GDP_percapita_fill_audit_2021.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

print("orig_gdp_path:", orig_gdp_path, orig_gdp_path.exists())
print("missing_path:", missing_path, missing_path.exists())
print("bea_dir:", bea_dir, bea_dir.exists())
print("co_est_path:", co_est_path, co_est_path.exists())
print("acs_path:", acs_path, acs_path.exists())

# helpers
def read_csv_fallback(path, **kwargs):
    last_err = None
    for enc in ["utf-8", "cp1252", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as e:
            last_err = e
    raise last_err

def clean_text(s):
    return (
        s.astype(str)
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
         .str.replace(r"\s+", " ", regex=True)
    )

def normalize_name(s):
    return (
        clean_text(s)
        .str.lower()
        .str.replace(r"\.", "", regex=True)
        .str.replace(r"'", "", regex=True)
        .str.replace(r"-", " ", regex=True)
        .str.replace(r"\bcounty\b", "", regex=True)
        .str.replace(r"\bcity\b", "", regex=True)
        .str.replace(r"\bmunicipality\b", "", regex=True)
        .str.replace(r"\bborough\b", "", regex=True)
        .str.replace(r"\bcensus area\b", "", regex=True)
        .str.replace(r"\bparish\b", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

def to_title_case(s):
    return clean_text(s).str.title()

# manual geoid patch 
MANUAL_GEOIDS = {
    ("alaska", "anchorage"): "02020",
    ("alaska", "denali"): "02068",
    ("alaska", "kodiak island"): "02150",
    ("alaska", "southeast fairbanks"): "02240",
    ("arizona", "mohave"): "04015",
    ("hawaii", "hawaii"): "15001",
    ("hawaii", "honolulu"): "15003",
    ("hawaii", "maui"): "15009",
    ("illinois", "dekalb"): "17037",
    ("illinois", "lasalle"): "17099",
    ("indiana", "newton"): "18111",
    ("iowa", "keokuk"): "19107",
    ("iowa", "o'brien"): "19141",
    ("kansas", "hamilton"): "20075",
    ("ohio", "ross"): "39141",
    ("puerto rico", "naguabo"): "72103",
    ("puerto rico", "santa isabel"): "72133",
    ("wisconsin", "fond du lac"): "55039",
}

PR_POP_OVERRIDES = {}

def load_population_2021():
    if co_est_path.exists():
        pop = read_csv_fallback(co_est_path, dtype=str)
        pop.columns = [c.strip() for c in pop.columns]

        required = {"STATE", "COUNTY", "POPESTIMATE2021"}
        if not required.issubset(pop.columns):
            raise ValueError(f"{co_est_path.name} is missing expected columns: {required}")

        pop["STATE"] = clean_text(pop["STATE"]).str.zfill(2)
        pop["COUNTY"] = clean_text(pop["COUNTY"]).str.zfill(3)
        pop = pop[pop["COUNTY"] != "000"].copy()

        pop["GEOID"] = pop["STATE"] + pop["COUNTY"]
        pop["population_2021"] = pd.to_numeric(pop["POPESTIMATE2021"], errors="coerce")

        keep = ["GEOID", "population_2021"]
        if "STNAME" in pop.columns:
            keep.append("STNAME")
        if "CTYNAME" in pop.columns:
            keep.append("CTYNAME")

        pop = pop[keep].drop_duplicates("GEOID").copy()

        if PR_POP_OVERRIDES:
            pr_pop = pd.DataFrame({
                "GEOID": list(PR_POP_OVERRIDES.keys()),
                "population_2021": list(PR_POP_OVERRIDES.values())
            })
            pop = pd.concat([pop, pr_pop], ignore_index=True)
            pop = pop.drop_duplicates("GEOID", keep="last")

        print("Using co-est2021-alldata.csv population file")
        return pop

    if acs_path.exists():
        pop = read_csv_fallback(acs_path, dtype=str)
        pop.columns = [c.strip() for c in pop.columns]

        geo_col = next((c for c in pop.columns if c.upper() == "GEO_ID"), None)
        pop_col = next((c for c in pop.columns if "B01003" in c and c.endswith("E")), None)

        if geo_col is None or pop_col is None:
            raise ValueError("ACS file format not recognized. Expected GEO_ID and B01003_001E-like column.")

        pop["GEOID"] = pop[geo_col].astype(str).str.extract(r"US(\d{5})")
        pop["population_2021"] = pd.to_numeric(pop[pop_col], errors="coerce")

        keep = ["GEOID", "population_2021"]
        if "NAME" in pop.columns:
            keep.append("NAME")

        pop = pop[keep].dropna(subset=["GEOID"]).drop_duplicates("GEOID").copy()

        if PR_POP_OVERRIDES:
            pr_pop = pd.DataFrame({
                "GEOID": list(PR_POP_OVERRIDES.keys()),
                "population_2021": list(PR_POP_OVERRIDES.values())
            })
            pop = pd.concat([pop, pr_pop], ignore_index=True)
            pop = pop.drop_duplicates("GEOID", keep="last")

        print("WARNING: using ACS fallback only")
        return pop

    raise FileNotFoundError("No population file found.")

# BEA GDP loader
def load_bea_county_gdp(bea_dir):
    bea_files = sorted(bea_dir.glob("CAGDP2*.csv"))
    if not bea_files:
        raise FileNotFoundError("No CAGDP2 CSV files found in data/CAGDP2/")

    frames = []
    for path in bea_files:
        try:
            df = read_csv_fallback(path, dtype=str)
        except Exception as e:
            print(f"Skipping {path.name}: {e}")
            continue

        needed = {"GeoFIPS", "GeoName", "Description", "2021"}
        if not needed.issubset(df.columns):
            continue

        df.columns = [c.strip() for c in df.columns]
        df["GeoFIPS"] = clean_text(df["GeoFIPS"]).str.replace('"', "", regex=False)
        df["GeoName"] = clean_text(df["GeoName"])
        df["Description"] = clean_text(df["Description"])

        df = df[
            (df["Description"] == "All industry total") &
            (df["GeoFIPS"].str.fullmatch(r"\d{5}", na=False)) &
            (~df["GeoFIPS"].str.endswith("000", na=False))
        ].copy()

        df["gdp_2021_thousands"] = pd.to_numeric(df["2021"], errors="coerce")
        df["source_file"] = path.name

        frames.append(df[["GeoFIPS", "GeoName", "gdp_2021_thousands", "source_file"]])

    if not frames:
        raise ValueError("No usable county GDP rows found in CAGDP2 files.")

    bea = pd.concat(frames, ignore_index=True)
    bea = bea.drop_duplicates(subset=["GeoFIPS"], keep="first").copy()
    return bea

VA_COMBINED_AREAS = {
    "51901": ["51003", "51540"],
    "51903": ["51005", "51580"],
    "51907": ["51015", "51790", "51820"],
    "51911": ["51031", "51680"],
    "51913": ["51035", "51640"],
    "51918": ["51053", "51570", "51730"],
    "51919": ["51059", "51600", "51610"],
    "51921": ["51069", "51840"],
    "51923": ["51081", "51595"],
    "51929": ["51089", "51690"],
    "51931": ["51095", "51830"],
    "51933": ["51121", "51750"],
    "51939": ["51143", "51590"],
    "51941": ["51149", "51670"],
    "51942": ["51153", "51683", "51685"],
    "51944": ["51161", "51775"],
    "51945": ["51163", "51530", "51678"],
    "51947": ["51165", "51660"],
    "51949": ["51175", "51620"],
    "51951": ["51177", "51630"],
    "51953": ["51191", "51520"],
    "51955": ["51195", "51720"],
    "51958": ["51199", "51735"],
}

va_crosswalk = pd.DataFrame(
    [(component_geoid, bea_geofips)
     for bea_geofips, components in VA_COMBINED_AREAS.items()
     for component_geoid in components],
    columns=["GEOID", "bea_geofips"]
)

orig_gdp = read_csv_fallback(orig_gdp_path, dtype=str)
orig_gdp.columns = [c.strip() for c in orig_gdp.columns]

rename_map = {}
if "State" in orig_gdp.columns:
    rename_map["State"] = "state"
if "County" in orig_gdp.columns:
    rename_map["County"] = "county"
if "county" not in orig_gdp.columns and "County Name" in orig_gdp.columns:
    rename_map["County Name"] = "county"
if "state" not in orig_gdp.columns and "STATE" in orig_gdp.columns:
    rename_map["STATE"] = "state"

orig_gdp = orig_gdp.rename(columns=rename_map)

required_orig = {"state", "county", "GDPpercapita"}
missing_required = required_orig - set(orig_gdp.columns)
if missing_required:
    raise ValueError(f"Original GDP file missing columns: {missing_required}")

if "Population Estimate" not in orig_gdp.columns:
    orig_gdp["Population Estimate"] = np.nan

missing = read_csv_fallback(missing_path, dtype=str)
missing.columns = [c.strip() for c in missing.columns]

missing["County Name"] = clean_text(missing["County Name"]).str.lower()
missing["State"] = clean_text(missing["State"]).str.lower()

if "GEOID" not in missing.columns:
    missing["GEOID"] = np.nan

missing["GEOID"] = missing["GEOID"].replace(r"^\s*$", np.nan, regex=True)
missing["GEOID"] = missing["GEOID"].replace({"nan": np.nan, "None": np.nan, "NaN": np.nan})

for (st, ct), geoid in MANUAL_GEOIDS.items():
    mask = (
        missing["GEOID"].isna() &
        (missing["State"] == st) &
        (missing["County Name"] == ct)
    )
    missing.loc[mask, "GEOID"] = geoid

missing["GEOID"] = missing["GEOID"].astype(str)
missing.loc[missing["GEOID"].isin(["nan", "None", "NaN"]), "GEOID"] = np.nan
missing.loc[missing["GEOID"].notna(), "GEOID"] = missing.loc[missing["GEOID"].notna(), "GEOID"].str.zfill(5)

pop = load_population_2021()
bea = load_bea_county_gdp(bea_dir)

print("Original GDP rows:", len(orig_gdp))
print("Missing counties list:", len(missing))
print("Population rows:", len(pop))
print("BEA county GDP rows:", len(bea))
print("Missing rows without GEOID after manual patch:", int(missing["GEOID"].isna().sum()))

# direct patch for rows with GEOID
direct_patch = (
    missing
    .merge(
        bea.rename(columns={"GeoFIPS": "GEOID", "GeoName": "bea_geoname_used"}),
        on="GEOID",
        how="left"
    )
    .merge(pop[["GEOID", "population_2021"]], on="GEOID", how="left")
)

direct_patch["GDPpercapita_fill"] = (
    direct_patch["gdp_2021_thousands"] * 1000 / direct_patch["population_2021"]
)
direct_patch["population_used_2021"] = direct_patch["population_2021"]
direct_patch["bea_geofips_used"] = direct_patch["GEOID"]

# virginia patch for rows without GEOID but in VA, using combined BEA areas
va_missing = missing[missing["State"] == "virginia"].copy()

va_area_pop = (
    va_crosswalk
    .merge(pop[["GEOID", "population_2021"]], on="GEOID", how="left")
    .groupby("bea_geofips", as_index=False)["population_2021"]
    .sum()
    .rename(columns={"population_2021": "population_used_2021"})
)

va_patch = (
    va_missing
    .merge(va_crosswalk, on="GEOID", how="left")
    .merge(
        bea.rename(columns={"GeoFIPS": "bea_geofips", "GeoName": "bea_geoname_used"}),
        on="bea_geofips",
        how="left"
    )
    .merge(va_area_pop, on="bea_geofips", how="left")
)

va_patch["GDPpercapita_fill"] = (
    va_patch["gdp_2021_thousands"] * 1000 / va_patch["population_used_2021"]
)
va_patch["bea_geofips_used"] = va_patch["bea_geofips"]

# combine direct patch and VA patch, prioritizing VA patch values where available
patch = (
    direct_patch[[
        "GEOID", "County Name", "State", "GDPpercapita_fill",
        "population_used_2021", "bea_geofips_used",
        "bea_geoname_used", "gdp_2021_thousands"
    ]]
    .merge(
        va_patch[[
            "GEOID", "GDPpercapita_fill", "population_used_2021",
            "bea_geofips_used", "bea_geoname_used", "gdp_2021_thousands"
        ]],
        on="GEOID",
        how="left",
        suffixes=("_direct", "_va")
    )
)

for col in [
    "GDPpercapita_fill",
    "population_used_2021",
    "bea_geofips_used",
    "bea_geoname_used",
    "gdp_2021_thousands",
]:
    patch[col] = patch[f"{col}_va"].combine_first(patch[f"{col}_direct"])

patch = patch[[
    "GEOID", "County Name", "State", "GDPpercapita_fill",
    "population_used_2021", "bea_geofips_used",
    "bea_geoname_used", "gdp_2021_thousands"
]].copy()

patch = patch[patch["GDPpercapita_fill"].notna()].copy()

# merge patch back to original GDP data
orig_gdp["state_key"] = normalize_name(orig_gdp["state"])
orig_gdp["county_key"] = normalize_name(orig_gdp["county"])

fill_rows = pd.DataFrame({
    "state": to_title_case(patch["State"]),
    "county": to_title_case(patch["County Name"]),
    "GDPpercapita_fill": patch["GDPpercapita_fill"],
    "Population Estimate Fill": patch["population_used_2021"],
    "GEOID": patch["GEOID"],
    "bea_geofips_used": patch["bea_geofips_used"],
    "bea_geoname_used": patch["bea_geoname_used"],
    "gdp_2021_thousands": patch["gdp_2021_thousands"],
})

fill_rows["state_key"] = normalize_name(fill_rows["state"])
fill_rows["county_key"] = normalize_name(fill_rows["county"])

merged = orig_gdp.merge(
    fill_rows[[
        "state_key", "county_key", "GDPpercapita_fill",
        "Population Estimate Fill", "GEOID",
        "bea_geofips_used", "bea_geoname_used", "gdp_2021_thousands"
    ]],
    on=["state_key", "county_key"],
    how="left"
)

merged["GDPpercapita"] = (
    pd.to_numeric(merged["GDPpercapita"], errors="coerce")
    .combine_first(pd.to_numeric(merged["GDPpercapita_fill"], errors="coerce"))
)

merged["Population Estimate"] = (
    pd.to_numeric(merged["Population Estimate"], errors="coerce")
    .combine_first(pd.to_numeric(merged["Population Estimate Fill"], errors="coerce"))
)

existing_keys = merged[["state_key", "county_key"]].drop_duplicates()

new_rows = (
    fill_rows
    .merge(existing_keys, on=["state_key", "county_key"], how="left", indicator=True)
    .query('_merge == "left_only"')
    .copy()
)

append_df = pd.DataFrame({
    "state": new_rows["state"],
    "county": new_rows["county"],
    "GDPpercapita": pd.to_numeric(new_rows["GDPpercapita_fill"], errors="coerce"),
    "Population Estimate": pd.to_numeric(new_rows["Population Estimate Fill"], errors="coerce"),
    "state_key": new_rows["state_key"],
    "county_key": new_rows["county_key"],
    "GEOID": new_rows["GEOID"],
    "bea_geofips_used": new_rows["bea_geofips_used"],
    "bea_geoname_used": new_rows["bea_geoname_used"],
    "gdp_2021_thousands": new_rows["gdp_2021_thousands"],
})

final = pd.concat([merged, append_df], ignore_index=True)

# export audit file showing which rows were filled and with what data
audit = (
    missing
    .merge(
        fill_rows[[
            "GEOID", "state", "county", "GDPpercapita_fill",
            "Population Estimate Fill", "bea_geofips_used",
            "bea_geoname_used", "gdp_2021_thousands"
        ]],
        on="GEOID",
        how="left"
    )
)

audit.to_csv(audit_path, index=False)

final_out = final[["state", "county", "GDPpercapita", "Population Estimate"]].copy()
final_out = final_out.sort_values(["state", "county"]).reset_index(drop=True)
final_out.to_csv(out_path, index=False)

print("\nSaved merged GDP file to:")
print(out_path)

print("\nSaved audit file to:")
print(audit_path)

print("\nFilled missing counties:", int(audit["GDPpercapita_fill"].notna().sum()), "out of", len(audit))

still_missing = audit[audit["GDPpercapita_fill"].isna()][["GEOID", "County Name", "State"]].copy()
still_missing = still_missing.sort_values(["State", "County Name"]).reset_index(drop=True)

print("Still missing after fill:", len(still_missing))
display(still_missing)

orig_gdp_path: ../Solar-NIMBY/data/GDP_percapita.csv True
missing_path: missing_gdp_per_capita_counties.csv True
bea_dir: data/CAGDP2 True
co_est_path: data/co-est2021-alldata.csv True
acs_path: data/ACSDT1Y2021.B01003_2026-03-12T195123/ACSDT1Y2021.B01003-Data.csv True
Using co-est2021-alldata.csv population file
Original GDP rows: 3041
Missing counties list: 77
Population rows: 3143
BEA county GDP rows: 4365
Missing rows without GEOID after manual patch: 0

Saved merged GDP file to:
data/processed/GDP_percapita_merged.csv

Saved audit file to:
data/processed/GDP_percapita_fill_audit_2021.csv

Filled missing counties: 74 out of 77
Still missing after fill: 3


,GEOID,County Name,State
0,15009,maui,hawaii
1,72103,naguabo,puerto rico
2,72133,santa isabel,puerto rico


In [7]:
# patches Maui from combined Maui + Kalawao

scores_root = Path(".")
bea_dir = scores_root / "data" / "CAGDP2"
co_est_path = scores_root / "data" / "co-est2021-alldata.csv"

out_path = scores_root / "data" / "processed" / "GDP_percapita_merged.csv"
audit_path = scores_root / "data" / "processed" / "GDP_percapita_fill_audit_2021.csv"

USE_PR_PROXY = False
PR_PROXY_GDP_PER_CAPITA = None 

def read_csv_fallback(path, **kwargs):
    last_err = None
    for enc in ["utf-8", "cp1252", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as e:
            last_err = e
    raise last_err

def clean_text(s):
    return (
        s.astype(str)
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
         .str.replace(r"\s+", " ", regex=True)
    )

def normalize_name(s):
    return (
        clean_text(s)
        .str.lower()
        .str.replace(r"\.", "", regex=True)
        .str.replace(r"'", "", regex=True)
        .str.replace(r"-", " ", regex=True)
        .str.replace(r"\bcounty\b", "", regex=True)
        .str.replace(r"\bcity\b", "", regex=True)
        .str.replace(r"\bmunicipality\b", "", regex=True)
        .str.replace(r"\bmunicipio\b", "", regex=True)
        .str.replace(r"\bborough\b", "", regex=True)
        .str.replace(r"\bcensus area\b", "", regex=True)
        .str.replace(r"\bparish\b", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

def first_nonnull(series):
    vals = series.dropna()
    return vals.iloc[0] if len(vals) else np.nan

# load current output files
final_out = read_csv_fallback(out_path, dtype=str)
audit = read_csv_fallback(audit_path, dtype=str)

final_out.columns = [c.strip() for c in final_out.columns]
audit.columns = [c.strip() for c in audit.columns]

# normalize keys
if "state" not in final_out.columns or "county" not in final_out.columns:
    raise ValueError("Expected final merged file to have columns: state, county")

final_out["state_key"] = normalize_name(final_out["state"])
final_out["county_key"] = normalize_name(final_out["county"])

audit["State"] = clean_text(audit["State"]).str.lower()
audit["County Name"] = clean_text(audit["County Name"]).str.lower()
audit["state_key"] = normalize_name(audit["State"])
audit["county_key"] = normalize_name(audit["County Name"])
audit["GEOID"] = clean_text(audit["GEOID"]).replace({"nan": np.nan, "None": np.nan})

# load county population file
pop = read_csv_fallback(co_est_path, dtype=str)
pop.columns = [c.strip() for c in pop.columns]
pop["STATE"] = clean_text(pop["STATE"]).str.zfill(2)
pop["COUNTY"] = clean_text(pop["COUNTY"]).str.zfill(3)
pop = pop[pop["COUNTY"] != "000"].copy()
pop["GEOID"] = pop["STATE"] + pop["COUNTY"]
pop["population_2021"] = pd.to_numeric(pop["POPESTIMATE2021"], errors="coerce")

# patch for Maui County using combined Maui + Kalawao BEA area
hi = read_csv_fallback(bea_dir / "CAGDP2_HI_2001_2021.csv", dtype=str)
hi.columns = [c.strip() for c in hi.columns]
hi["GeoFIPS"] = clean_text(hi["GeoFIPS"]).str.replace('"', "", regex=False)
hi["GeoName"] = clean_text(hi["GeoName"])
hi["GeoName_norm"] = normalize_name(hi["GeoName"])
hi["Description"] = clean_text(hi["Description"])
hi["2021_num"] = pd.to_numeric(hi["2021"], errors="coerce")

maui_bea = hi[
    (hi["Description"] == "All industry total") &
    (
        hi["GeoName_norm"].str.contains("maui", na=False) &
        hi["GeoName_norm"].str.contains("kalawao", na=False)
    )
].copy()

if len(maui_bea) == 0:
    # fallback: any all-industry row mentioning Maui
    maui_bea = hi[
        (hi["Description"] == "All industry total") &
        (hi["GeoName_norm"].str.contains("maui", na=False))
    ].copy()

if len(maui_bea) == 0:
    raise ValueError("Could not find a Maui-related BEA row in CAGDP2_HI_2001_2021.csv")

maui_gdp_2021_thousands = first_nonnull(maui_bea["2021_num"])
maui_bea_geofips = first_nonnull(maui_bea["GeoFIPS"])
maui_bea_name = first_nonnull(maui_bea["GeoName"])

# Maui County (15009) + Kalawao County (15005)
maui_pop = pop.loc[pop["GEOID"].isin(["15009", "15005"]), "population_2021"].sum()

if pd.isna(maui_gdp_2021_thousands) or pd.isna(maui_pop) or maui_pop == 0:
    raise ValueError("Could not compute Maui GDP per capita from BEA + population file")

maui_gdp_pc = maui_gdp_2021_thousands * 1000 / maui_pop

patch_rows = [
    {
        "GEOID": "15009",
        "state_key": "hawaii",
        "county_key": "maui",
        "state": "Hawaii",
        "county": "Maui",
        "GDPpercapita_fill": maui_gdp_pc,
        "Population Estimate Fill": maui_pop,
        "bea_geofips_used": maui_bea_geofips,
        "bea_geoname_used": maui_bea_name,
        "gdp_2021_thousands": maui_gdp_2021_thousands,
        "fill_note": "Used combined Maui + Kalawao BEA area"
    }
]

# inspect Puerto Rico rows in BEA file to see if we can fill any missing municipios there, using the same approach as above
pr_path = bea_dir / "CAGDP2_PORT_2001_2021.csv"
pr_candidates = []

if pr_path.exists():
    pr = read_csv_fallback(pr_path, dtype=str)
    pr.columns = [c.strip() for c in pr.columns]
    pr["GeoFIPS"] = clean_text(pr["GeoFIPS"]).str.replace('"', "", regex=False)
    pr["GeoName"] = clean_text(pr["GeoName"])
    pr["GeoName_norm"] = normalize_name(pr["GeoName"])
    pr["Description"] = clean_text(pr["Description"])
    pr["2021_num"] = pd.to_numeric(pr["2021"], errors="coerce")

    pr_targets = [
        ("72103", "naguabo"),
        ("72133", "santa isabel"),
    ]

    for geoid, county_name in pr_targets:
        matches = pr[
            (pr["Description"] == "All industry total") &
            (
                pr["GeoFIPS"].eq(geoid) |
                pr["GeoName_norm"].str.contains(county_name, na=False)
            )
        ].copy()

        if len(matches) > 0:
            gdp_2021_thousands = first_nonnull(matches["2021_num"])
            bea_geofips = first_nonnull(matches["GeoFIPS"])
            bea_geoname = first_nonnull(matches["GeoName"])

            if USE_PR_PROXY and PR_PROXY_GDP_PER_CAPITA is not None:
                patch_rows.append({
                    "GEOID": geoid,
                    "state_key": "puerto rico",
                    "county_key": county_name,
                    "state": "Puerto Rico",
                    "county": county_name.title(),
                    "GDPpercapita_fill": PR_PROXY_GDP_PER_CAPITA,
                    "Population Estimate Fill": np.nan,
                    "bea_geofips_used": bea_geofips,
                    "bea_geoname_used": bea_geoname,
                    "gdp_2021_thousands": gdp_2021_thousands,
                    "fill_note": "Used Puerto Rico proxy GDP per capita"
                })
        else:
            pr_candidates.append((geoid, county_name, "NOT FOUND IN CAGDP2_PORT"))

else:
    pr_candidates = [
        ("72103", "naguabo", "PORT FILE MISSING"),
        ("72133", "santa isabel", "PORT FILE MISSING"),
    ]

patch_df = pd.DataFrame(patch_rows)

final_out["GDPpercapita"] = pd.to_numeric(final_out["GDPpercapita"], errors="coerce")
final_out["Population Estimate"] = pd.to_numeric(final_out["Population Estimate"], errors="coerce")

final_out = final_out.merge(
    patch_df[[
        "state_key", "county_key",
        "GDPpercapita_fill", "Population Estimate Fill"
    ]],
    on=["state_key", "county_key"],
    how="left"
)

final_out["GDPpercapita"] = final_out["GDPpercapita"].combine_first(
    pd.to_numeric(final_out["GDPpercapita_fill"], errors="coerce")
)
final_out["Population Estimate"] = final_out["Population Estimate"].combine_first(
    pd.to_numeric(final_out["Population Estimate Fill"], errors="coerce")
)

# append patch rows if not already present
existing_keys = set(zip(final_out["state_key"], final_out["county_key"]))
to_append = patch_df[
    ~patch_df.apply(lambda r: (r["state_key"], r["county_key"]) in existing_keys, axis=1)
].copy()

if len(to_append):
    append_rows = pd.DataFrame({
        "state": to_append["state"],
        "county": to_append["county"],
        "GDPpercapita": pd.to_numeric(to_append["GDPpercapita_fill"], errors="coerce"),
        "Population Estimate": pd.to_numeric(to_append["Population Estimate Fill"], errors="coerce"),
        "state_key": to_append["state_key"],
        "county_key": to_append["county_key"],
        "GDPpercapita_fill": to_append["GDPpercapita_fill"],
        "Population Estimate Fill": to_append["Population Estimate Fill"],
    })
    final_out = pd.concat([final_out, append_rows], ignore_index=True)

final_out_export = final_out[["state", "county", "GDPpercapita", "Population Estimate"]].copy()
final_out_export = final_out_export.sort_values(["state", "county"]).reset_index(drop=True)
final_out_export.to_csv(out_path, index=False)

if "GDPpercapita_fill" in audit.columns:
    audit["GDPpercapita_fill"] = pd.to_numeric(audit["GDPpercapita_fill"], errors="coerce")
if "Population Estimate Fill" in audit.columns:
    audit["Population Estimate Fill"] = pd.to_numeric(audit["Population Estimate Fill"], errors="coerce")

audit = audit.merge(
    patch_df[[
        "GEOID", "GDPpercapita_fill", "Population Estimate Fill",
        "bea_geofips_used", "bea_geoname_used", "gdp_2021_thousands", "fill_note"
    ]],
    on="GEOID",
    how="left",
    suffixes=("", "_patch")
)

for col in [
    "GDPpercapita_fill",
    "Population Estimate Fill",
    "bea_geofips_used",
    "bea_geoname_used",
    "gdp_2021_thousands",
]:
    if col in audit.columns and f"{col}_patch" in audit.columns:
        audit[col] = audit[col].combine_first(audit[f"{col}_patch"])

if "fill_note" not in audit.columns:
    audit["fill_note"] = np.nan
if "fill_note_patch" in audit.columns:
    audit["fill_note"] = audit["fill_note"].combine_first(audit["fill_note_patch"])

drop_cols = [c for c in audit.columns if c.endswith("_patch")]
audit = audit.drop(columns=drop_cols)

audit.to_csv(audit_path, index=False)

still_missing = audit[audit["GDPpercapita_fill"].isna()][["GEOID", "County Name", "State"]].copy()
still_missing = still_missing.sort_values(["State", "County Name"]).reset_index(drop=True)

print("Patched rows:")
display(patch_df[[
    "GEOID", "state", "county", "GDPpercapita_fill",
    "Population Estimate Fill", "bea_geofips_used", "bea_geoname_used", "fill_note"
]])

print("\nPuerto Rico lookup status:")
display(pd.DataFrame(pr_candidates, columns=["GEOID", "county", "status"]))

print("\nRe-saved:")
print(out_path)
print(audit_path)

print("\nStill missing after safe final patch:", len(still_missing))
display(still_missing)

Patched rows:


,GEOID,state,county,GDPpercapita_fill,Population Estimate Fill,bea_geofips_used,bea_geoname_used,fill_note
0,15009,Hawaii,Maui,63030.005539,164303,15901,"Maui + Kalawao, HI*",Used combined Maui + Kalawao BEA area



Puerto Rico lookup status:


,GEOID,county,status
0,72103,naguabo,NOT FOUND IN CAGDP2_PORT
1,72133,santa isabel,NOT FOUND IN CAGDP2_PORT



Re-saved:
data/processed/GDP_percapita_merged.csv
data/processed/GDP_percapita_fill_audit_2021.csv

Still missing after safe final patch: 2


,GEOID,County Name,State
0,72103,naguabo,puerto rico
1,72133,santa isabel,puerto rico
